# Juego de la Vida de Conway

LEAD university

Programación Paralela y Distribuida

Implementación Paralela con Análisis de Complejidad

Kristhel Porras Mata  
Profesor: Johansell Villalobos Cubillo

## Instalación y configuración

In [2]:
!pip install numpy matplotlib numba -q
print("✓ Dependencias instaladas")

✓ Dependencias instaladas


## Importaciones

In [31]:
import time
import tracemalloc
import os

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from numba import njit, prange
from PIL import Image

print("✓ Módulos importados")

✓ Módulos importados


## Módulo 1: Núcleo de cálculo

Funciones compiladas con Numba para calcular generaciones.

In [4]:
@njit(parallel=True, cache=True)
def calcular_generacion_paralela(grilla: np.ndarray) -> np.ndarray:
    """Calcula la siguiente generación usando paralelización.

    Utiliza prange para distribuir el cálculo entre múltiples núcleos.
    Cada celda cuenta sus 8 vecinos y aplica las reglas de Conway.

    Args:
        grilla: Matriz actual (int8, 0=muerto, 1=vivo)

    Returns:
        Matriz de la siguiente generación
    """
    filas, columnas = grilla.shape
    grilla_nueva = np.empty_like(grilla)

    for fila in prange(filas):
        for columna in range(columnas):
            vecinos = (
                grilla[(fila - 1) % filas, (columna - 1) % columnas]
                + grilla[(fila - 1) % filas,  columna             ]
                + grilla[(fila - 1) % filas, (columna + 1) % columnas]
                + grilla[ fila,             (columna - 1) % columnas]
                + grilla[ fila,             (columna + 1) % columnas]
                + grilla[(fila + 1) % filas, (columna - 1) % columnas]
                + grilla[(fila + 1) % filas,  columna             ]
                + grilla[(fila + 1) % filas, (columna + 1) % columnas]
            )
            celda = grilla[fila, columna]

            if celda == 1:
                grilla_nueva[fila, columna] = 1 if (vecinos == 2 or vecinos == 3) else 0
            else:
                grilla_nueva[fila, columna] = 1 if vecinos == 3 else 0

    return grilla_nueva


@njit(cache=True)
def calcular_generacion_secuencial(grilla: np.ndarray) -> np.ndarray:
    """Calcula la siguiente generación de forma secuencial.

    Implementa la misma lógica sin paralelización.
    Se usa como referencia para medir speedup.

    Args:
        grilla: Matriz actual

    Returns:
        Matriz de la siguiente generación
    """
    filas, columnas = grilla.shape
    grilla_nueva = np.empty_like(grilla)

    for fila in range(filas):
        for columna in range(columnas):
            vecinos = (
                grilla[(fila - 1) % filas, (columna - 1) % columnas]
                + grilla[(fila - 1) % filas,  columna             ]
                + grilla[(fila - 1) % filas, (columna + 1) % columnas]
                + grilla[ fila,             (columna - 1) % columnas]
                + grilla[ fila,             (columna + 1) % columnas]
                + grilla[(fila + 1) % filas, (columna - 1) % columnas]
                + grilla[(fila + 1) % filas,  columna             ]
                + grilla[(fila + 1) % filas, (columna + 1) % columnas]
            )
            celda = grilla[fila, columna]

            if celda == 1:
                grilla_nueva[fila, columna] = 1 if (vecinos == 2 or vecinos == 3) else 0
            else:
                grilla_nueva[fila, columna] = 1 if vecinos == 3 else 0

    return grilla_nueva

print("✓ Núcleo de cálculo definido")

✓ Núcleo de cálculo definido


## Módulo 2: Patrones clásicos

In [5]:
PATRONES = {
    "planeador": np.array(
        [[0, 1, 0],
         [0, 0, 1],
         [1, 1, 1]],
        dtype=np.int8
    ),
    "parpadeador": np.array(
        [[1, 1, 1]],
        dtype=np.int8
    ),
    "sapo": np.array(
        [[0, 1, 1, 1],
         [1, 1, 1, 0]],
        dtype=np.int8
    ),
}

def colocar_patron(grilla: np.ndarray, patron: np.ndarray, fila: int, columna: int) -> None:
    """Inserta un patrón en la grilla.

    Args:
        grilla: Grilla destino
        patron: Patrón a insertar
        fila: Fila de inicio
        columna: Columna de inicio
    """
    filas_patron, columnas_patron = patron.shape
    grilla[fila:fila + filas_patron, columna:columna + columnas_patron] = patron

print("✓ Patrones definidos: planeador, parpadeador, sapo")

✓ Patrones definidos: planeador, parpadeador, sapo


## Módulo 3: Clase JuegoVida

In [6]:
class JuegoVida:
    """Simulador del Juego de la Vida de Conway.

    Proporciona interfaz para crear grillas, ejecutar simulaciones
    e insertar patrones clásicos.
    """

    def __init__(self, filas: int = 64, columnas: int = 64,
                 inicializacion="aleatorio", semilla: int = 42, paralelo: bool = True):
        """Inicializa una simulación.

        Args:
            filas: Número de filas
            columnas: Número de columnas
            inicializacion: 'aleatorio', 'vacío' o array de numpy
            semilla: Semilla para aleatoriedad
            paralelo: Usar paralelización
        """
        self.filas = filas
        self.columnas = columnas
        self.paralelo = paralelo
        self.generacion = 0

        if isinstance(inicializacion, np.ndarray):
            self.grilla = inicializacion.astype(np.int8)
        elif inicializacion == "aleatorio":
            generador = np.random.default_rng(semilla)
            self.grilla = generador.integers(0, 2, size=(filas, columnas), dtype=np.int8)
        elif inicializacion == "vacío":
            self.grilla = np.zeros((filas, columnas), dtype=np.int8)
        else:
            raise ValueError(f"Modo de inicialización desconocido: '{inicializacion}'")

        calcular_generacion_paralela(np.zeros((4, 4), dtype=np.int8))
        calcular_generacion_secuencial(np.zeros((4, 4), dtype=np.int8))

    def paso(self) -> None:
        """Avanza una generación."""
        if self.paralelo:
            self.grilla = calcular_generacion_paralela(self.grilla)
        else:
            self.grilla = calcular_generacion_secuencial(self.grilla)
        self.generacion += 1

    def ejecutar(self, pasos: int) -> None:
        """Ejecuta múltiples generaciones.

        Args:
            pasos: Número de generaciones a simular
        """
        for _ in range(pasos):
            self.paso()

    def obtener_estado(self) -> np.ndarray:
        """Retorna copia del estado actual.

        Returns:
            Copia de la grilla
        """
        return self.grilla.copy()

    def colocar(self, nombre_patron: str, fila: int = 2, columna: int = 2) -> None:
        """Coloca un patrón en la grilla.

        Args:
            nombre_patron: Nombre del patrón
            fila: Fila de inicio
            columna: Columna de inicio
        """
        patron = PATRONES[nombre_patron]
        colocar_patron(self.grilla, patron, fila, columna)

print("✓ Clase JuegoVida definida")

✓ Clase JuegoVida definida


## Módulo 4: Animaciones

In [36]:
def generar_animacion(nombre_patron: str, tamaño_grilla: int = 64,
                     fotogramas: int = 60, intervalo: int = 100,
                     ruta_salida: str = None):
    """Crea y guarda una animación GIF."""

    juego = JuegoVida(tamaño_grilla, tamaño_grilla, inicializacion="vacío")
    juego.colocar(nombre_patron, fila=4, columna=4)
    fotogramas_lista = []

    for generacion_actual in range(fotogramas):
        figura, eje = plt.subplots(figsize=(5, 5), facecolor="#f5f1ed")
        eje.set_facecolor("#f5f1ed")
        eje.axis("off")

        eje.imshow(juego.obtener_estado(), interpolation="nearest",
                  cmap="twilight", vmin=0, vmax=1)
        eje.set_title(
            f"{nombre_patron.capitalize()}  Generación {juego.generacion}  {tamaño_grilla}x{tamaño_grilla}",
            color="#5a4a42", fontsize=10, pad=6, fontfamily="serif"
        )

        plt.tight_layout(pad=0)
        figura.canvas.draw()

        # LÍNEA CORREGIDA
        ancho, alto = figura.canvas.get_width_height()
        buffer = figura.canvas.buffer_rgba()
        imagen = Image.frombytes('RGBA', (ancho, alto), buffer)
        imagen = imagen.convert('RGB')
        fotogramas_lista.append(imagen)

        plt.close(figura)
        juego.paso()

    if ruta_salida and fotogramas_lista:
        fotogramas_lista[0].save(
            ruta_salida,
            save_all=True,
            append_images=fotogramas_lista[1:],
            duration=intervalo,
            loop=0
        )
        print(f"Animación guardada: {ruta_salida}")

print("✓ Función de animación definida")

✓ Función de animación definida


## Ejecutar animaciones

In [38]:
print("Generando animaciones...\n")

generar_animacion('planeador', 64, 80, 120, ruta_salida='patron_planeador.gif')
generar_animacion('parpadeador', 32, 30, 200, ruta_salida='patron_parpadeador.gif')
generar_animacion('sapo', 64, 40, 200, ruta_salida='patron_sapo.gif')

print("\nAnimaciones completadas")

Generando animaciones...

Animación guardada: patron_planeador.gif
Animación guardada: patron_parpadeador.gif
Animación guardada: patron_sapo.gif

Animaciones completadas


## Módulo 5: Benchmark

In [16]:
def ejecutar_benchmark(tamaños=None, pasos=10):
    """Mide rendimiento para diferentes tamaños de grilla.

    Args:
        tamaños: Lista de tamaños a testear
        pasos: Número de generaciones por tamaño

    Returns:
        Diccionario con resultados
    """
    if tamaños is None:
        tamaños = [32, 64, 128, 256, 512, 768, 1024]

    tiempo_paralelo, tiempo_secuencial, memoria_mb = [], [], []

    print("\nBenchmark en progreso...\n")
    print(f"{'Grilla':<12} {'Paralelo':<12} {'Secuencial':<12} {'Memoria':<12}")
    print("-" * 48)

    for n in tamaños:
        generador = np.random.default_rng(0)
        grilla = generador.integers(0, 2, size=(n, n), dtype=np.int8)

        juego = JuegoVida(n, n, inicializacion=grilla.copy(), paralelo=True)
        inicio = time.perf_counter()
        juego.ejecutar(pasos)
        tiempo_paralelo.append((time.perf_counter() - inicio) / pasos)

        juego2 = JuegoVida(n, n, inicializacion=grilla.copy(), paralelo=False)
        inicio = time.perf_counter()
        juego2.ejecutar(pasos)
        tiempo_secuencial.append((time.perf_counter() - inicio) / pasos)

        tracemalloc.start()
        juego3 = JuegoVida(n, n, inicializacion=grilla.copy(), paralelo=True)
        juego3.ejecutar(1)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        memoria_mb.append(pico / 1024 / 1024)

        print(f"{n}x{n:<8} {tiempo_paralelo[-1]*1000:>10.2f} ms  {tiempo_secuencial[-1]*1000:>10.2f} ms  {memoria_mb[-1]:>10.3f} MB")

    return {
        "tamaños": np.array(tamaños),
        "tiempo_paralelo": np.array(tiempo_paralelo),
        "tiempo_secuencial": np.array(tiempo_secuencial),
        "memoria_mb": np.array(memoria_mb),
    }

datos = ejecutar_benchmark()
print("\nBenchmark completado")


Benchmark en progreso...

Grilla       Paralelo     Secuencial   Memoria     
------------------------------------------------
32x32             0.16 ms        0.03 ms       0.003 MB
64x64             0.28 ms        0.09 ms       0.009 MB
128x128            0.65 ms        0.28 ms       0.032 MB
256x256            2.63 ms        0.97 ms       0.126 MB
512x512           10.30 ms        3.83 ms       0.503 MB
768x768           25.90 ms        8.56 ms       1.126 MB
1024x1024          46.42 ms       32.58 ms       2.001 MB

Benchmark completado


## Módulo 6: Gráficas

In [25]:
def generar_graficas(datos, prefijo_salida='rendimiento'):
    """Genera tres gráficas de análisis de rendimiento.

    Args:
        datos: Diccionario con resultados del benchmark
        prefijo_salida: Prefijo para los archivos PNG
    """
    tamaños = datos["tamaños"]
    t_par = datos["tiempo_paralelo"]
    t_sec = datos["tiempo_secuencial"]
    memoria = datos["memoria_mb"]

    aceleracion = t_sec / t_par
    nucleos = os.cpu_count() or 1
    eficiencia = aceleracion / nucleos

    FONDO = "#f5f1ed"
    AZUL = "#a8d8ea"
    VERDE = "#b4e7b4"
    ROJO = "#ffb3ba"
    ORO = "#FF7518"
    TEXTO = "#5a4a42"
    BORDE = "#e0d5cf"
    GRILLA = "#d5cac0"

    def estilizar_eje(eje, titulo, etiqueta_x, etiqueta_y):
        """Aplica estilo pastel a un gráfico."""
        eje.set_facecolor(FONDO)
        eje.set_title(titulo, color=TEXTO, fontsize=11, pad=8,
                     fontfamily="serif", fontweight="bold")
        eje.set_xlabel(etiqueta_x, color=TEXTO, fontsize=9, fontfamily="sans-serif")
        eje.set_ylabel(etiqueta_y, color=TEXTO, fontsize=9, fontfamily="sans-serif")
        eje.tick_params(colors=TEXTO, labelsize=8)
        for espina in eje.spines.values():
            espina.set_edgecolor(BORDE)
            espina.set_linewidth(0.8)
        eje.grid(color=GRILLA, linestyle="--", linewidth=0.5, alpha=0.6)

    figura1, eje = plt.subplots(figsize=(7, 4.5), facecolor=FONDO)
    estilizar_eje(eje, "Tiempo de Ejecución vs Tamaño de Grilla",
                 "Lado de la Grilla (n)", "Tiempo por Generación (segundos)")
    eje.plot(tamaños, t_par, "o-", color=AZUL, lw=2, label="Paralelo (Numba)", markersize=5)
    eje.plot(tamaños, t_sec, "s--", color=ROJO, lw=2, label="Secuencial (Numba)", markersize=5)
    eje.legend(facecolor=FONDO, edgecolor=BORDE, labelcolor=TEXTO, fontsize=9)
    plt.tight_layout()
    figura1.savefig(f'{prefijo_salida}_lineal.png', dpi=130, bbox_inches="tight", facecolor=FONDO)
    plt.close(figura1)
    print(f"Gráfica guardada: {prefijo_salida}_lineal.png")

    figura2, eje = plt.subplots(figsize=(7, 4.5), facecolor=FONDO)
    estilizar_eje(eje, "Análisis de Complejidad: Escala Log-Log",
                 "log(Lado de la Grilla n)", "log(Tiempo en segundos)")
    eje.loglog(tamaños, t_par, "o-", color=AZUL, lw=2, label="Paralelo (Numba)", markersize=5)
    eje.loglog(tamaños, t_sec, "s--", color=ROJO, lw=2, label="Secuencial (Numba)", markersize=5)

    escala_oro = t_par[len(t_par)//2] / (tamaños[len(tamaños)//2] ** 2)
    eje.loglog(tamaños, escala_oro * tamaños**2, ":", color=ORO, lw=1.5, label="O(n²) Referencia")

    eje.legend(facecolor=FONDO, edgecolor=BORDE, labelcolor=TEXTO, fontsize=9)
    plt.tight_layout()
    figura2.savefig(f'{prefijo_salida}_loglog.png', dpi=130, bbox_inches="tight", facecolor=FONDO)
    plt.close(figura2)
    print(f"Gráfica guardada: {prefijo_salida}_loglog.png")

    figura3, ejes = plt.subplots(1, 2, figsize=(11, 4.5), facecolor=FONDO)

    eje_izq = ejes[0]
    estilizar_eje(eje_izq, "Métricas de Paralelización",
                 "Lado de la Grilla (n)", "Aceleración S / Eficiencia E")
    eje_izq.plot(tamaños, aceleracion, "o-", color=AZUL, lw=2, label="Aceleración S", markersize=5)
    eje_izq.plot(tamaños, eficiencia, "s--", color=VERDE, lw=2, label="Eficiencia E", markersize=4)
    eje_izq.axhline(1.0, color=BORDE, linestyle=":", lw=1)
    eje_izq.legend(facecolor=FONDO, edgecolor=BORDE, labelcolor=TEXTO, fontsize=8)

    eje_der = ejes[1]
    estilizar_eje(eje_der, "Consumo de Memoria",
                 "Lado de la Grilla (n)", "Memoria Pico (MB)")
    eje_der.fill_between(tamaños, memoria, alpha=0.3, color=ROJO)
    eje_der.plot(tamaños, memoria, "o-", color=ROJO, lw=2, markersize=5, label="Memoria Pico")

    escala_mem = memoria[-1] / (tamaños[-1] ** 2)
    eje_der.plot(tamaños, escala_mem * tamaños**2, ":", color=ORO, lw=1.5, label="O(n²) Referencia")
    eje_der.legend(facecolor=FONDO, edgecolor=BORDE, labelcolor=TEXTO, fontsize=8)

    plt.tight_layout()
    figura3.savefig(f'{prefijo_salida}_aceleracion_memoria.png', dpi=130,
                   bbox_inches="tight", facecolor=FONDO)
    plt.close(figura3)
    print(f"Gráfica guardada: {prefijo_salida}_aceleracion_memoria.png")

print("\nGenerando gráficas...\n")
generar_graficas(datos, prefijo_salida='rendimiento')
print("\nAnálisis completado")


Generando gráficas...

Gráfica guardada: rendimiento_lineal.png
Gráfica guardada: rendimiento_loglog.png
Gráfica guardada: rendimiento_aceleracion_memoria.png

Análisis completado


In [ ]:
## Verificación final

In [27]:
import os

archivos = [
    'patron_planeador.gif',
    'patron_parpadeador.gif',
    'patron_sapo.gif',
    'rendimiento_lineal.png',
    'rendimiento_loglog.png',
    'rendimiento_aceleracion_memoria.png'
]

print("Archivos generados:\n")
for archivo in archivos:
    if os.path.exists(archivo):
        tamaño = os.path.getsize(archivo) / 1024
        print(f"✓ {archivo:<35} ({tamaño:.1f} KB)")

Archivos generados:

✓ patron_planeador.gif                (92.0 KB)
✓ patron_parpadeador.gif              (38.3 KB)
✓ patron_sapo.gif                     (46.4 KB)
✓ rendimiento_lineal.png              (58.9 KB)
✓ rendimiento_loglog.png              (62.5 KB)
✓ rendimiento_aceleracion_memoria.png (96.3 KB)
